# 00. Полный интерактивный workshop notebook

Один notebook вместо шести: конфигурация реальной LLM, создание Deep Agents graph,
connectors, отправка connector payload в OpenRouter и финальный SWE graph.

Секреты читаются из `.env` и не печатаются. В outputs сохранены только модель,
ответы и usage без API key.

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import httpx
from deepagents import create_deep_agent
from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'pyproject.toml').exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / '.env')

DEFAULT_MODEL = 'openrouter:openai/gpt-oss-120b:free'

def model_name() -> str:
    return os.getenv('OPENCLAW_MODEL', DEFAULT_MODEL)

REPO_ROOT=/Users/ken/work/openclaw clone
MODEL=openrouter:openai/gpt-oss-120b:free
OPENROUTER_API_KEY_SET=True


## 1. Как подключается реальная LLM

В `.env` нужны две переменные:

```bash
OPENROUTER_API_KEY=...
OPENCLAW_MODEL=openrouter:openai/gpt-oss-120b:free
```

В коде мы убираем префикс `openrouter:` только для прямого HTTP API.
Для Deep Agents entrypoint строка `openrouter:...` остаётся удобной модельной строкой LangChain.

In [2]:
OPENROUTER_MODEL = model_name().removeprefix('openrouter:')

def call_openrouter(prompt: str, *, max_tokens: int = 160) -> dict:
    api_key = os.environ['OPENROUTER_API_KEY']
    payload = {
        'model': OPENROUTER_MODEL,
        'messages': [{'role': 'user', 'content': prompt}],
        'max_tokens': max_tokens,
    }
    headers = {
        'Authorization': f'Bearer {api_key}',
        'Content-Type': 'application/json',
        'HTTP-Referer': 'http://localhost/openclaw-workshop',
        'X-Title': 'OpenClaw Workshop',
    }
    with httpx.Client(timeout=45) as client:
        response = client.post(
            'https://openrouter.ai/api/v1/chat/completions',
            headers=headers,
            json=payload,
        )
    response.raise_for_status()
    data = response.json()
    return {
        'model': data.get('model'),
        'content': data['choices'][0]['message']['content'],
        'usage': data.get('usage', {}),
    }

print(f'REPO_ROOT={REPO_ROOT}')
print(f'MODEL={model_name()}')
print(f'OPENROUTER_API_KEY_SET={bool(os.getenv("OPENROUTER_API_KEY"))}')

REPO_ROOT=/Users/ken/work/openclaw clone
MODEL=openrouter:openai/gpt-oss-120b:free
OPENROUTER_API_KEY_SET=True


In [3]:
llm_result = call_openrouter(
    'Ответь одним коротким предложением по-русски: что такое Deep Agent?',
    max_tokens=120,
)
print(json.dumps(llm_result, ensure_ascii=False, indent=2))

{
  "model": "openai/gpt-oss-120b:free",
  "content": "Deep Agent — это интеллектуальная система, обучающаяся самостоятельно решать задачи через взаимодействие с окружающей средой.",
  "usage": {
    "prompt_tokens": 84,
    "completion_tokens": 43,
    "total_tokens": 127,
    "cost": 0,
    "is_byok": false,
    "prompt_tokens_details": {
      "cached_tokens": 64,
      "cache_write_tokens": 0,
      "audio_tokens": 0,
      "video_tokens": 0
    },
    "cost_details": {
      "upstream_inference_cost": 0,
      "upstream_inference_prompt_cost": 0,
      "upstream_inference_completions_cost": 0
    },
    "completion_tokens_details": {
      "reasoning_tokens": 8,
      "image_tokens": 0,
      "audio_tokens": 0
    }
  }
}


## 2. Минимальный Deep Agents graph

Граф создаётся локально. Для live demo модельный HTTP helper выше надёжнее,
потому что текущий `langchain_openrouter/openrouter` SDK иногда зависает на transport layer,
тогда как прямой OpenRouter HTTP endpoint отвечает корректно.

In [4]:
BASE_PROMPT = """\
You are SberClaw, an experimental open-source coding agent built with LangChain
Deep Agents. You help with software engineering, repository navigation, product
research, and implementation.

Work like a careful senior engineer:
- inspect before editing;
- keep changes focused;
- verify with tests or equivalent checks;
- explain only the useful result to the user.
"""

agent = create_deep_agent(
    model=model_name(),
    tools=[],
    system_prompt=BASE_PROMPT,
)
print(type(agent).__name__)

CompiledStateGraph


## 3. Connector payload → prompt → real LLM

In [5]:
from connectors.demo import get_demo_issue, list_demo_issues

issues = list_demo_issues.invoke({})
issue = get_demo_issue.invoke({'issue_id': 'DEMO-101'})
print(issues)
print(issue)

[
  {
    "id": "DEMO-101",
    "title": "Add connector onboarding flow",
    "status": "ready",
    "priority": "high",
    "owner": "workshop",
    "summary": "Show how a connector becomes an agent tool without changing the Deep Agents runtime."
  },
  {
    "id": "DEMO-102",
    "title": "Enable local shell mode guardrails",
    "status": "backlog",
    "priority": "medium",
    "owner": "platform",
    "summary": "Document the risks and add an approval checklist before enabling shell mode."
  }
]
{
  "id": "DEMO-101",
  "title": "Add connector onboarding flow",
  "status": "ready",
  "priority": "high",
  "owner": "workshop",
  "summary": "Show how a connector becomes an agent tool without changing the Deep Agents runtime."
}


In [6]:
issue_prompt = (
    'Ты агент на воркшопе. По данным issue ниже предложи следующий инженерный шаг. '
    'Ответь по-русски, в 2 пункта.\n\n'
    f'{issue}'
)
issue_llm_result = call_openrouter(issue_prompt, max_tokens=220)
print(json.dumps(issue_llm_result, ensure_ascii=False, indent=2))

{
  "model": "openai/gpt-oss-120b:free",
  "content": "1. **Разработать отдельный модуль‑обёртку (adapter) для коннектора**, который будет реализовывать унифицированный интерфейс «инструмента агента». Этот модуль должен принимать входные данные от коннектора, преобразовывать их в формат, ожидаемый Deep Agents, и возвращать результат в виде сообщений/событий, понятных рантайму. При этом сам Deep Agents runtime остаётся неизменным – взаимодействие происходит через четко определённый API адаптера.\n\n2. **Создать и задокументировать процесс onboarding‑flow**:  \n   - подготовить шаблон конфигурации коннектора (параметры, типы данных, схема авторизации);  \n   - реализовать автоматическую валидацию конфигурации и регистрацию коннектора в реестре инструментов агента;  \n   - добавить примеры и тесты, демонстрирующие подключение нового коннектора без модификации ядра.  \n\nЭти шаги позволят быстро интегрировать любые коннекторы как инструменты агента, сохранив стабильность текущей среды выпо

## 4. Telegram connector в безопасном dry-run

In [7]:
from connectors.telegram import send_telegram_message

telegram_result = send_telegram_message.invoke(
    {
        'message': 'OpenClaw notebooks can call connectors and a real LLM.',
        'dry_run': True,
    }
)
print(telegram_result)

{
  "dry_run": true,
  "method": "sendMessage",
  "payload": {
    "chat_id": "<missing TELEGRAM_CHAT_ID>",
    "text": "OpenClaw notebooks can call connectors and a real LLM.",
    "disable_web_page_preview": true
  }
}


## 5. Финальный graph с connectors, skills, memory и SWE mode

In [8]:
from connectors import CONNECTOR_TOOLS

SUBAGENTS = [
    {
        'name': 'repo-researcher',
        'description': 'Map repository structure, APIs, tests, and likely change locations.',
        'system_prompt': 'Inspect files and return concise findings with paths.',
    },
    {
        'name': 'reviewer',
        'description': 'Review proposed patches for bugs, missing tests, and regressions.',
        'system_prompt': 'Focus on correctness, edge cases, tests, security, and maintainability.',
    },
]

swe_prompt = BASE_PROMPT + """\

You are running in SWE mode. Reproduce, localize, patch, test, summarize.
"""

swe_agent = create_deep_agent(
    model=model_name(),
    tools=CONNECTOR_TOOLS,
    system_prompt=swe_prompt,
    subagents=SUBAGENTS,
    skills=['/skills/swe-resolution'],
    memory=['/AGENTS.md'],
    interrupt_on={'execute': True, 'write_file': True, 'edit_file': True},
)
print(type(swe_agent).__name__)

CompiledStateGraph


## Что показывать голосом

1. `.env` задаёт provider/model/key; секреты не лежат в notebook.
2. `call_openrouter` делает настоящий запрос к модели и показывает usage.
3. Connector tools дают структурированный payload.
4. Этот payload отправляется в LLM как контекст задачи.
5. Deep Agents graph собирает тот же набор capabilities в runnable agent.